# Kenzie 360 — Modelo de risco de não resolução

**O que este notebook faz:** olha o que se sabe no primeiro minuto de uma conversa e estima a chance de a Kenzie **não** conseguir resolver aquele atendimento sozinha.

**Como rodar:** execute as células na ordem, de cima para baixo. Cada uma imprime o resultado.

**Quem escreveu:** grupo Kenzie 360 — MBA Engenharia de Dados, Mackenzie.

## 0. Preparação

Se alguma biblioteca faltar, tire o `#` da linha abaixo e rode uma vez.

In [1]:
# !pip install pandas scikit-learn google-cloud-bigquery db-dtypes

import hashlib, warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
print("bibliotecas carregadas")

bibliotecas carregadas


## 1. De onde vêm os dados

Duas opções. **BigQuery** lê a `vw_wide_atendimento` da camada gold — é o caminho de produção.
**CSV** lê o arquivo local, para quem não tem acesso ao projeto na nuvem.

Troque `FONTE` conforme o seu caso.

In [2]:
FONTE = "bigquery"          # troque para "csv" se não tiver acesso ao GCP

PROJETO = "kenzie-360-mba"
REGIAO  = "us-central1"
CAMINHO_CSV = "../2_dados/atual_v8/dataset_kenzie360_atendimentos_v8.csv"

if FONTE == "bigquery":
    from google.cloud import bigquery
    # A Wide Table nomeia duas colunas de outro jeito e nao expoe data_hora.
    # Os apelidos abaixo devolvem os nomes que o resto do notebook espera.
    sql = f"""
        SELECT
          id_conversa, id_cliente, segmento_cliente, arquetipo,
          perfil_tecnologico, faixa_etaria, idioma, canal_entrada,
          categoria_assunto, produto_relacionado, contatos_previos,
          origem_contato                                AS origem,
          hora_do_dia                                   AS hora,
          MOD(EXTRACT(DAYOFWEEK FROM data_ref) + 5, 7)  AS dia_semana,
          resolvido_bot,
          reabertura,
          -- NAO sao caracteristicas: o modelo nunca ve estas tres colunas.
          -- Elas so aparecem no backtest (secao 11), para conferir se a nota
          -- de risco tambem ordena tempo e abandono.
          duracao_min, num_mensagens, fila_abandonada
        FROM `{PROJETO}.kenzie360_gold.vw_wide_atendimento`
    """
    df = bigquery.Client(project=PROJETO).query(sql, location=REGIAO).to_dataframe()

else:
    COLUNAS = ["id_conversa","id_cliente","segmento_cliente","arquetipo","perfil_tecnologico",
               "faixa_etaria","idioma","origem","canal_entrada","categoria_assunto",
               "produto_relacionado","contatos_previos","data_hora","resolvido_bot","reabertura",
               "duracao_seg","num_mensagens","transferencia_iniciada","humano_atendeu"]
    df = pd.read_csv(CAMINHO_CSV, sep=";", encoding="utf-8-sig", low_memory=False)[COLUNAS]
    df["data_hora"]  = pd.to_datetime(df["data_hora"], errors="coerce")
    df["hora"]       = df["data_hora"].dt.hour
    df["dia_semana"] = df["data_hora"].dt.dayofweek
    # mesmas tres colunas de backtest, reconstruidas a partir do CSV cru
    df["duracao_min"]     = pd.to_numeric(df["duracao_seg"], errors="coerce") / 60
    df["fila_abandonada"] = (df["transferencia_iniciada"].astype(str).str.lower().isin(["true","1","sim"])
                             & ~df["humano_atendeu"].astype(str).str.lower().isin(["true","1","sim"]))

print(f"{len(df):,} conversas carregadas".replace(",", "."))
df.head(3)

36.000 conversas carregadas


,id_conversa,id_cliente,segmento_cliente,arquetipo,perfil_tecnologico,faixa_etaria,idioma,canal_entrada,categoria_assunto,produto_relacionado,contatos_previos,origem,hora,dia_semana,resolvido_bot,reabertura,duracao_min,num_mensagens,fila_abandonada
0,CONV-013681,CLI-68704AB8E154,CDE,Alta Renda/Private,medio,adulto,ES,WhatsApp,Investimentos CDB,CDB,2,Receptivo,12,2,False,False,15.63,14,False
1,CONV-013719,CLI-F44B708DAC76,CDE,Alta Renda/Private,medio,adulto,ES,WhatsApp,Onboarding - Documentacao,Conta Corrente,17,Receptivo,15,2,False,False,17.72,20,False
2,CONV-013634,CLI-E1E4F3398344,Correntista Nacional,Alta Renda/Private,medio,adulto,PT,WhatsApp,Onboarding - Documentacao,Conta Corrente,0,Receptivo,8,2,True,False,2.47,6,False


## 2. O que é uma regressão logística

Antes de rodar, vale entender o que a conta faz — em uma frase:

> **É uma soma de pontos.** Cada característica da conversa vale um número de pontos, positivo ou negativo. No fim, o total vira uma probabilidade entre 0 e 1.

É a mesma matemática de um **score de crédito**. Lá, cada informação do cadastro soma ou subtrai pontos, e o total vira a chance de inadimplência. Aqui, cada característica da conversa soma ou subtrai, e o total vira a chance de a Kenzie não resolver.

| Característica da conversa | Pontos |
|---|---|
| Assunto é Bloqueio de TED/PIX | soma muito |
| Assunto é Onboarding | soma |
| Assunto é Saldo e Extrato | subtrai muito |
| Cliente já procurou o banco várias vezes | soma um pouco |

A única diferença para uma soma comum é o último passo: a conta "espreme" o total para caber entre 0 e 1, porque probabilidade não pode ser 150% nem negativa.

**Por que essa e não outra:** cada característica sai com o seu peso visível. Dá para abrir o modelo e mostrar *por que* uma conversa recebeu nota alta — coisa que um modelo mais sofisticado não entrega.

## 3. Quem entra na conta

Tiramos duas categorias: **Disparo Ativo** (mensagem que o banco envia, não é demanda do cliente) e **Ruído** (engano, número errado, "oi" sem contexto).

Nenhuma das duas vira atendimento. Se ficassem, o modelo ganharia acerto de graça só por separar "isto não é uma demanda" — que é uma decisão que ninguém precisa de modelo para tomar.

In [3]:
FORA = ["Disparo Ativo", "Ruido / Nao-atendimento"]

antes = len(df)
df = df[~df["categoria_assunto"].isin(FORA)].reset_index(drop=True)
print(f"excluídas : {antes - len(df):,}".replace(",", "."))
print(f"restam    : {len(df):,} demandas reais".replace(",", "."))

excluídas : 4.958
restam    : 31.042 demandas reais


## 4. O que queremos prever

O alvo é **o bot NÃO resolveu**. Escolhemos prever a falha, e não o sucesso, porque é a falha que gera uma ação: encaminhar para um humano.

In [4]:
resolvido = df["resolvido_bot"].astype(str).str.strip().str.lower().isin(["true", "1", "sim"])
y = (~resolvido).astype(int)

print(f"a Kenzie NÃO resolve em {y.mean()*100:.1f}% das conversas")
print(f"a Kenzie resolve sozinha em {(1-y.mean())*100:.1f}%")

a Kenzie NÃO resolve em 70.7% das conversas
a Kenzie resolve sozinha em 29.3%


## 5. O que o modelo pode olhar

**Treze características**, todas conhecidas quando a intenção do cliente é identificada. Nenhuma delas depende do desfecho da conversa.

Duas decisões de modelagem que valem registro:

- **`reabertura`** entrou nesta versão. Ela marca se a conversa **anterior daquele mesmo cliente** ficou sem resolução — é passado puro, verificado no gerador (`.shift()` olha só para trás). Não é redundante com a recorrência: em todos os níveis de contatos anteriores, ser reabertura piora o desfecho.
- **`contatos_previos` entra em faixas**, não como número puro. A regressão logística trata número puro como reta, mas o efeito não é reta (65,3% → 82,1% de falha). Quebrando em faixas o modelo consegue curvar. Isso não acrescenta variável — muda só como aquela variável entra.

**Controle de vazamento:** 18 colunas ficaram proibidas por lista explícita — duração, número de mensagens, status, transferência, sentimento. Todas só existem depois que a conversa acabou.

In [5]:
# recorrência em faixas: o efeito não é linear, então o número puro subaproveita a variável
df["contatos_faixa"] = pd.cut(pd.to_numeric(df["contatos_previos"], errors="coerce").fillna(0),
                              [-1, 0, 1, 2, 4, 8, 999],
                              labels=["0", "1", "2", "3-4", "5-8", "9+"]).astype(str)

CATEGORICAS = ["segmento_cliente","arquetipo","perfil_tecnologico","faixa_etaria",
               "idioma","origem","canal_entrada","categoria_assunto","produto_relacionado",
               "reabertura","contatos_faixa"]
NUMERICAS   = ["hora","dia_semana"]

for c in CATEGORICAS:
    df[c] = df[c].fillna("(vazio)").astype(str).str.strip().str.lower() if c == "reabertura" else df[c].fillna("(vazio)").astype(str)
for c in NUMERICAS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-1)

X = df[CATEGORICAS + NUMERICAS]
print(f"{len(CATEGORICAS + NUMERICAS)} variáveis, todas conhecidas na abertura:")
print(" ·", "  ·  ".join(CATEGORICAS + NUMERICAS))
print()
print("reabertura na base:")
print(df["reabertura"].value_counts().to_string())
print()
print("recorrência em faixas:")
print(df["contatos_faixa"].value_counts().reindex(["0","1","2","3-4","5-8","9+"]).to_string())


13 variáveis, todas conhecidas na abertura:
 · segmento_cliente  ·  arquetipo  ·  perfil_tecnologico  ·  faixa_etaria  ·  idioma  ·  origem  ·  canal_entrada  ·  categoria_assunto  ·  produto_relacionado  ·  reabertura  ·  contatos_faixa  ·  hora  ·  dia_semana

reabertura na base:
reabertura
false    25555
true      5487

recorrência em faixas:
contatos_faixa
0      12898
1       5194
2       3691
3-4     4284
5-8     2845
9+      2130


## 6. Separar treino e teste — **por cliente**

O modelo aprende com uma parte dos dados e é avaliado na outra, que ele nunca viu.

A separação é **por cliente, não por conversa**. Quem volta ao banco tem conversas parecidas; separando por conversa, o mesmo cliente cairia dos dois lados e o resultado pareceria melhor do que é.

O `md5` só serve para sortear de forma estável: o mesmo cliente cai sempre no mesmo lado, toda vez que o notebook roda.

In [6]:
balde  = df["id_cliente"].astype(str).map(lambda s: int(hashlib.md5(s.encode()).hexdigest(), 16) % 100)
treino = balde < 75
teste  = balde >= 75

print(f"treino : {treino.sum():,} conversas de {df.loc[treino,'id_cliente'].nunique():,} clientes".replace(",", "."))
print(f"teste  : {teste.sum():,} conversas de {df.loc[teste,'id_cliente'].nunique():,} clientes".replace(",", "."))
print(f"clientes nos dois lados: {len(set(df.loc[treino,'id_cliente']) & set(df.loc[teste,'id_cliente']))}")

treino : 23.330 conversas de 10.372 clientes
teste  : 7.712 conversas de 3.375 clientes
clientes nos dois lados: 0


## 7. Treinar

O `OneHotEncoder` transforma texto em números — "Câmbio" vira uma coluna que é 1 quando o assunto é Câmbio e 0 quando não é. É só tradução; nenhuma decisão acontece aqui.

In [7]:
preparo = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATEGORICAS),
    ("numericas",   "passthrough", NUMERICAS)])

modelo = Pipeline([("preparo", preparo), ("regressao", LogisticRegression(max_iter=3000))])
modelo.fit(X[treino], y[treino])

risco = modelo.predict_proba(X[teste])[:, 1]
print("modelo treinado")
print(f"nota de risco calculada para {len(risco):,} conversas de teste".replace(",", "."))

modelo treinado
nota de risco calculada para 7.712 conversas de teste


## 8. O resultado, e como ler cada número

**Baseline** — o acerto de quem não tem modelo nenhum, chutando sempre a resposta mais comum. É a régua: sem ela, qualquer número parece bom.

**Acurácia** — de cada 100 conversas, quantas o modelo classifica certo. Só faz sentido comparada ao baseline.

**AUC** — pega uma conversa que a Kenzie resolveu e uma que ela não resolveu, sorteadas ao acaso: é a chance de o modelo ter dado nota de risco maior para a que realmente falhou. 0,5 é moeda; 1,0 é perfeito. É a mesma métrica do Gini usado em crédito (`Gini = 2 × AUC − 1`).

In [8]:
baseline  = max(y[teste].mean(), 1 - y[teste].mean())
acuracia  = accuracy_score(y[teste], (risco >= 0.5).astype(int))
auc       = roc_auc_score(y[teste], risco)

print(f"baseline  : {baseline*100:.1f}%   (chutar sempre 'não resolve')")
print(f"acurácia  : {acuracia*100:.1f}%")
print(f"AUC       : {auc:.4f}     (Gini = {2*auc-1:.4f})")

baseline  : 70.5%   (chutar sempre 'não resolve')
acurácia  : 72.8%
AUC       : 0.6729     (Gini = 0.3457)


## 9. A matriz de confusão

A acurácia é um número só, e ele esconde **que tipo** de erro o modelo comete. A matriz de confusão abre isso em quatro caixas.

Traduzindo para a operação, onde "prever não resolve" significa **mandar a conversa direto para o humano**:

| | previu **resolve** (deixa com a Kenzie) | previu **não resolve** (manda pro humano) |
|---|---|---|
| **o bot resolveu** | Verdadeiro Negativo — certo | **Falso Positivo** — ocupei um atendente à toa |
| **o bot não resolveu** | **Falso Negativo** — o cliente esperou à toa | Verdadeiro Positivo — acertei |

Os dois erros custam coisas diferentes, e é por isso que existem duas métricas:

- **Precisão** — das que mandei para o humano, quantas realmente precisavam? Controla o **Falso Positivo** (custo de atendente).
- **Recall (sensibilidade)** — das que precisavam de humano, quantas eu peguei? Controla o **Falso Negativo** (cliente esperando).

Não dá para maximizar as duas ao mesmo tempo. Escolher entre elas é decisão de negócio, não de estatística.

In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

previsto = (risco >= 0.50).astype(int)
vn, fp, fn, vp = confusion_matrix(y[teste], previsto).ravel()

print("MATRIZ DE CONFUSÃO — limiar 0,50 (o padrão do scikit-learn)\n")
print(f"{'':<26}{'previu RESOLVE':>16}{'previu NÃO RESOLVE':>21}")
print(f"{'bot RESOLVEU':<26}{vn:>10} (VN){fp:>16} (FP)")
print(f"{'bot NÃO RESOLVEU':<26}{fn:>10} (FN){vp:>16} (VP)")

print(f"\nacurácia       {accuracy_score(y[teste], previsto)*100:5.1f}%   dos acertos no total")
print(f"precisão       {precision_score(y[teste], previsto)*100:5.1f}%   das que mandei pro humano, quantas precisavam")
print(f"recall         {recall_score(y[teste], previsto)*100:5.1f}%   das que precisavam, quantas eu peguei")
print(f"F1             {f1_score(y[teste], previsto)*100:5.1f}%   média harmônica das duas")
print(f"especificidade {vn/(vn+fp)*100:5.1f}%   das que a Kenzie resolveria, quantas deixei com ela")

print(f"\nAUC        {auc:.4f}   (moeda = 0,500)")
print(f"PR-AUC     {average_precision_score(y[teste], risco):.4f}   (baseline = {y[teste].mean():.4f}, a própria taxa de falha)")

print(f"\nLEITURA: neste limiar o modelo manda {(vp+fp)/len(previsto)*100:.0f}% de TODAS as conversas para o humano.")
print("Recall altíssimo, especificidade péssima — ele quase não confia na Kenzie.")
print("Como regra de decisão isso é inútil: 0,50 não é um limiar escolhido, é o default da biblioteca.")

MATRIZ DE CONFUSÃO — limiar 0,50 (o padrão do scikit-learn)

                            previu RESOLVE   previu NÃO RESOLVE
bot RESOLVEU                     462 (VN)            1813 (FP)
bot NÃO RESOLVEU                 283 (FN)            5154 (VP)

acurácia        72.8%   dos acertos no total
precisão        74.0%   das que mandei pro humano, quantas precisavam
recall          94.8%   das que precisavam, quantas eu peguei
F1              83.1%   média harmônica das duas
especificidade  20.3%   das que a Kenzie resolveria, quantas deixei com ela

AUC        0.6729   (moeda = 0,500)
PR-AUC     0.8131   (baseline = 0.7050, a própria taxa de falha)

LEITURA: neste limiar o modelo manda 90% de TODAS as conversas para o humano.
Recall altíssimo, especificidade péssima — ele quase não confia na Kenzie.
Como regra de decisão isso é inútil: 0,50 não é um limiar escolhido, é o default da biblioteca.


## 10. Escolher o limiar

O modelo não devolve "sim" ou "não". Devolve uma **probabilidade** entre 0 e 1. Transformar isso em decisão exige um corte — o **limiar** — e esse corte é uma escolha nossa.

O 0,50 da célula anterior não foi escolhido por ninguém: é o default. E ele não serve, porque manda 89% da fila para o humano.

A tabela abaixo varre os limiares possíveis. Ler assim: **subir o limiar aumenta a precisão e derruba o recall.** Mando menos gente para o humano, erro menos, mas deixo mais cliente esperando.

In [10]:
print(f"{'limiar':>7}{'enviados':>10}{'% da fila':>11}{'precisão':>10}{'recall':>9}{'F1':>8}{'VP':>7}{'FP':>7}{'FN':>7}")
print("-"*76)
for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    yp = (risco >= t).astype(int)
    if yp.sum() == 0:
        continue
    vn_, fp_, fn_, vp_ = confusion_matrix(y[teste], yp, labels=[0, 1]).ravel()
    print(f"{t:7.2f}{vp_+fp_:10d}{(vp_+fp_)/len(yp)*100:10.1f}%"
          f"{precision_score(y[teste], yp)*100:9.1f}%{recall_score(y[teste], yp)*100:8.1f}%"
          f"{f1_score(y[teste], yp)*100:7.1f}%{vp_:7d}{fp_:7d}{fn_:7d}")

print("\nO MESMO, EM MOEDA DE NEGÓCIO")
print("cada VP evita 13,1 min de espera do cliente | cada FP ocupa um atendente sem necessidade\n")
print(f"{'limiar':>7}{'% da fila':>11}{'horas evitadas':>16}{'falsos alarmes':>16}{'precisão':>10}")
print("-"*60)
for t in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
    yp = (risco >= t).astype(int)
    vn_, fp_, fn_, vp_ = confusion_matrix(y[teste], yp, labels=[0, 1]).ravel()
    if vp_ + fp_ == 0:
        continue
    print(f"{t:7.2f}{(vp_+fp_)/len(yp)*100:10.1f}%{vp_*13.1/60:15.0f}h{fp_:16d}"
          f"{vp_/(vp_+fp_)*100:9.1f}%")

print("\nO QUE ESTA TABELA MOSTRA")
print("A precisão sobe pouco (74% -> 85%) enquanto o recall despenca (94% -> 16%).")
print("Isso é assinatura de modelo de AUC modesto: não existe limiar mágico que dê")
print("precisão alta sem abrir mão de quase tudo. Então o limiar não sai da estatística —")
print("sai da CAPACIDADE da operação: quantas conversas por dia o time humano aguenta?")

 limiar  enviados  % da fila  precisão   recall      F1     VP     FP     FN
----------------------------------------------------------------------------
   0.50      6967      90.3%     74.0%    94.8%   83.1%   5154   1813    283
   0.55      6810      88.3%     74.4%    93.2%   82.7%   5067   1743    370
   0.60      6344      82.3%     75.5%    88.1%   81.4%   4792   1552    645
   0.65      5634      73.1%     77.1%    79.9%   78.5%   4346   1288   1091
   0.70      4318      56.0%     79.3%    63.0%   70.2%   3426    892   2011
   0.75      3554      46.1%     81.0%    53.0%   64.1%   2880    674   2557
   0.80      2536      32.9%     82.9%    38.7%   52.7%   2102    434   3335
   0.85      1159      15.0%     85.8%    18.3%   30.1%    994    165   4443
   0.90       370       4.8%     87.6%     6.0%   11.2%    324     46   5113

O MESMO, EM MOEDA DE NEGÓCIO
cada VP evita 13,1 min de espera do cliente | cada FP ocupa um atendente sem necessidade

 limiar  % da fila  horas evitada

## 11. Um modelo mais complexo melhora?

Se uma random forest — bem mais sofisticada — der praticamente o mesmo resultado, isso diz que **o limite é a informação disponível, e não o algoritmo**. E justifica ficar no modelo simples, que a gente consegue explicar.

In [11]:
floresta = Pipeline([("preparo", preparo),
                     ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=50,
                                                   random_state=42, n_jobs=-1))])
floresta.fit(X[treino], y[treino])
auc_rf = roc_auc_score(y[teste], floresta.predict_proba(X[teste])[:, 1])

print(f"AUC da regressão logística : {auc:.3f}")
print(f"AUC da random forest       : {auc_rf:.3f}")
print()
print("praticamente igual" if abs(auc - auc_rf) < 0.02 else "diferença relevante — investigar")

AUC da regressão logística : 0.673
AUC da random forest       : 0.669

praticamente igual


## 12. Quais características pesam mais (os coeficientes)

Aqui fica visível **por que** uma conversa recebe nota alta. Peso positivo aumenta o risco de a Kenzie não resolver; negativo diminui.

In [12]:
nomes = modelo.named_steps["preparo"].get_feature_names_out()
pesos = modelo.named_steps["regressao"].coef_[0]
ordem = np.argsort(pesos)
limpo = lambda n: n.replace("categoricas__", "").replace("numericas__", "")

print("AUMENTAM o risco de a Kenzie não resolver")
for i in ordem[::-1][:6]:
    print(f"   {limpo(nomes[i]):<46}{pesos[i]:+.2f}")
print()
print("DIMINUEM")
for i in ordem[:5]:
    print(f"   {limpo(nomes[i]):<46}{pesos[i]:+.2f}")

AUMENTAM o risco de a Kenzie não resolver
   categoria_assunto_Bloqueio TED/PIX            +0.94
   categoria_assunto_Onboarding - Documentacao   +0.71
   contatos_faixa_5-8                            +0.49
   contatos_faixa_9+                             +0.41
   produto_relacionado_Conta Corrente            +0.29
   reabertura_true                               +0.28

DIMINUEM
   categoria_assunto_Saldo e Extrato             -1.38
   contatos_faixa_0                              -0.37
   categoria_assunto_Investimentos CDB           -0.33
   produto_relacionado_CDB                       -0.33
   contatos_faixa_1                              -0.26


## 13. Quais variáveis realmente pesam

Os coeficientes da seção anterior dizem *para que lado* cada característica empurra. Não dizem se ela **faz falta**.

O teste correto é tirar uma variável, treinar de novo e medir quanto o AUC cai. O que não derruba nada, não está fazendo nada.

*(esta célula treina 13 modelos — leva cerca de meio minuto)*

In [13]:
def auc_sem(cols):
    pr = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore", min_frequency=20),
         [c for c in cols if c in CATEGORICAS]),
        ("numericas", "passthrough", [c for c in cols if c not in CATEGORICAS])])
    m = Pipeline([("preparo", pr), ("regressao", LogisticRegression(max_iter=3000))])
    m.fit(df.loc[treino, cols], y[treino])
    return roc_auc_score(y[teste], m.predict_proba(df.loc[teste, cols])[:, 1])

todas = CATEGORICAS + NUMERICAS
completo = auc_sem(todas)
perdas = sorted(((c, completo - auc_sem([x for x in todas if x != c])) for c in todas),
                key=lambda r: -r[1])

print(f"AUC com todas as 12 variáveis: {completo:.4f}\n")
print(f"{'variável':<24}{'queda no AUC ao tirar':>23}   classificação")
print("-"*68)
for c, d in perdas:
    classe = "ESSENCIAL" if d >= 0.010 else ("útil" if d >= 0.002 else "irrelevante")
    print(f"{c:<24}{d:>+23.4f}   {classe}")

print("\nE SE FICARMOS SÓ COM AS QUE IMPORTAM?")
for cols, rot in [(["categoria_assunto"], "só o assunto (1 variável)"),
                  (["categoria_assunto", "contatos_previos"], "assunto + recorrência (2 variáveis)"),
                  (todas, "as 12 atuais")]:
    print(f"   {rot:<38} AUC {auc_sem(cols):.4f}")

print("\nLEITURA: duas variáveis entregam o mesmo que doze. As outras dez não são")
print("prejudiciais, mas também não são o modelo — são contexto. Reportar isso é")
print("mais honesto do que exibir doze variáveis como se todas trabalhassem.")

AUC com todas as 12 variáveis: 0.6729

variável                  queda no AUC ao tirar   classificação
--------------------------------------------------------------------
categoria_assunto                       +0.0692   ESSENCIAL
contatos_faixa                          +0.0116   ESSENCIAL
reabertura                              +0.0039   útil
hora                                    +0.0007   irrelevante
segmento_cliente                        +0.0001   irrelevante
perfil_tecnologico                      +0.0001   irrelevante
faixa_etaria                            +0.0001   irrelevante
idioma                                  -0.0000   irrelevante
dia_semana                              -0.0000   irrelevante
canal_entrada                           -0.0001   irrelevante
produto_relacionado                     -0.0001   irrelevante
origem                                  -0.0001   irrelevante
arquetipo                               -0.0003   irrelevante

E SE FICARMOS SÓ COM AS QUE IMPO

## 14. O teste que vale: o que aconteceu de fato

Dividimos as conversas de teste em cinco faixas de risco e olhamos **o que realmente aconteceu** em cada uma. É o mesmo método com que um banco valida um score de crédito antes de usar.

**Como o corte é feito:** `pd.qcut` divide por **quintis** — cinco grupos do mesmo tamanho, ~1.542 conversas cada. O corte é **relativo**, não um limiar fixo: a fronteira sai da distribuição das notas, não de uma regra de negócio. A célula imprime as probabilidades onde cada faixa começa e termina, para o corte ficar explícito.

Faixa é para **diagnosticar** (o score ordena?); limiar é para **decidir** (mando ou não mando). São coisas diferentes — a seção 10 trata da segunda.

Repare que abandono e duração o modelo nunca viu.

In [14]:
T = df[teste].copy()
T["risco"]  = risco
T["falhou"] = y[teste].values
T["faixa"]  = pd.qcut(T["risco"], 5, labels=["1 menor risco","2","3","4","5 maior risco"])

resumo = T.groupby("faixa", observed=True).apply(lambda g: pd.Series({
    "conversas"           : len(g),
    "risco medio %"       : g["risco"].mean() * 100,
    "bot resolveu %"      : (1 - g["falhou"].mean()) * 100,
    "duracao media (min)" : pd.to_numeric(g["duracao_min"], errors="coerce").mean(),
    "mensagens"           : pd.to_numeric(g["num_mensagens"], errors="coerce").mean(),
    "abandono na fila %"  : g["fila_abandonada"].astype(float).mean() * 100}))
print(resumo.round(1).to_string())

cortes = T.groupby("faixa", observed=True)["risco"].agg(["min", "max"])
print("\nONDE CADA FAIXA COMEÇA E TERMINA (probabilidade prevista)")
print(cortes.round(3).to_string())
print("\nAs três últimas colunas o modelo NUNCA viu. Se piorarem da faixa 1 para a 5,")
print("a nota está ordenando tempo e abandono sem ter sido treinada para isso.")


               conversas  risco medio %  bot resolveu %  duracao media (min)  mensagens  abandono na fila %
faixa                                                                                                      
1 menor risco     1543.0           48.6            50.7                 11.1       11.8                 2.5
2                 1542.0           65.8            33.8                 13.6       12.8                 3.6
3                 1543.0           72.9            26.9                 13.5       12.7                 5.3
4                 1541.0           80.8            21.0                 14.5       13.1                 5.2
5 maior risco     1543.0           87.7            15.2                 14.7       13.2                 4.7

ONDE CADA FAIXA COMEÇA E TERMINA (probabilidade prevista)
                 min    max
faixa                      
1 menor risco  0.277  0.613
2              0.613  0.689
3              0.689  0.778
4              0.778  0.835
5 maior risco  0.

## 15. As três faixas operacionais

As cinco faixas da seção anterior são **quintis** — cinco grupos do mesmo tamanho, recalculados a cada modelo. Servem para provar que o score ordena.

Para operar, a etiqueta precisa de significado **estável**: "faixa Alto" tem que querer dizer a mesma coisa em setembro e em dezembro. Por isso as faixas operacionais usam **cortes fixos de probabilidade**, escolhidos pelo time:

| Faixa | Nota de risco |
|---|---|
| **Baixo** | até 0,65 |
| **Médio** | 0,65 a 0,80 |
| **Alto** | acima de 0,80 |

Elas têm três usos, e **nenhum deles é decidir o roteamento** — quem decide é a cota por hora:

1. **Etiqueta no painel** — "Alto" lê melhor que "0,87"
2. **Leitura de negócio** — quanto a Kenzie resolve em cada faixa
3. **Monitor de deriva** — se a distribuição entre as faixas mudar muito no lote novo, alguma coisa mudou no mundo e é hora de reavaliar o modelo

**Uma ressalva de linguagem:** como a base inteira falha em 70,5%, "Baixo" aqui significa *baixo em relação ao resto*, não baixo em absoluto. Mesmo na faixa Baixo a Kenzie ainda deixa de resolver cerca de metade.

In [15]:
CORTE_BAIXO, CORTE_ALTO = 0.65, 0.80

T["faixa_op"] = np.where(T["risco"] < CORTE_BAIXO, "Baixo",
                  np.where(T["risco"] < CORTE_ALTO, "Médio", "Alto"))

ordem = ["Baixo", "Médio", "Alto"]
op = (T.groupby("faixa_op", observed=True)
        .apply(lambda g: pd.Series({
            "conversas"        : len(g),
            "% da fila"        : len(g) / len(T) * 100,
            "Kenzie resolve %" : (1 - g["falhou"].mean()) * 100,
            "falha %"          : g["falhou"].mean() * 100,
            "duração (min)"    : pd.to_numeric(g["duracao_min"], errors="coerce").mean()}))
        .reindex(ordem))
print(op.round(1).to_string())

print("\nGUARDE ESTES TRÊS NÚMEROS — são a referência para o monitor de deriva:")
for f in ordem:
    print(f"   {f:6s} {op.loc[f, '% da fila']:5.1f}% da fila")
print("\nSe no lote novo a distribuição sair muito destes valores, é sinal de que")
print("a população mudou — e o modelo precisa ser reavaliado.")

          conversas  % da fila  Kenzie resolve %  falha %  duração (min)
faixa_op                                                                
Baixo        2078.0       26.9              47.5     52.5           11.7
Médio        3098.0       40.2              27.6     72.4           13.7
Alto         2536.0       32.9              17.1     82.9           14.8

GUARDE ESTES TRÊS NÚMEROS — são a referência para o monitor de deriva:
   Baixo   26.9% da fila
   Médio   40.2% da fila
   Alto    32.9% da fila

Se no lote novo a distribuição sair muito destes valores, é sinal de que
a população mudou — e o modelo precisa ser reavaliado.


## 16. A fila de correção

Onde a Kenzie mais falha, ordenado por **quanto aquilo custa** — volume vezes taxa de falha. Não é modelo: é uma soma. E é o que diz ao time de produto qual fluxo do bot reescrever primeiro.

In [16]:
fila = (df.assign(falhou=y)
          .groupby("categoria_assunto")
          .agg(volume=("id_conversa", "size"), taxa_falha=("falhou", "mean"), falhas=("falhou", "sum"))
          .reset_index())
fila["taxa_falha"] *= 100
fila = fila.sort_values("falhas", ascending=False)
fila["% do total"] = fila["falhas"] / fila["falhas"].sum() * 100
fila["acumulado"]  = fila["% do total"].cumsum()

print(fila.round(1).to_string(index=False))

                categoria_assunto  volume  taxa_falha  falhas  % do total  acumulado
                 Bloqueio TED/PIX    4433        85.7    3801        17.3       17.3
        Onboarding - Documentacao    4457        82.6    3681        16.8       34.1
                           Cambio    4665        74.2    3461        15.8       49.8
                  Primeiro Acesso    3819        68.2    2606        11.9       61.7
             Alteracao de Limites    3390        73.0    2476        11.3       73.0
                Cartao de Credito    3422        63.8    2184         9.9       82.9
         Termos e Documentacao BR    2320        70.7    1640         7.5       90.4
                Investimentos CDB    2278        50.6    1152         5.2       95.6
                  Saldo e Extrato    2024        38.9     788         3.6       99.2
Operacoes PJ - Folha e Pagamentos     234        72.2     169         0.8      100.0


## 17. Guardar o resultado

Duas saídas: a nota de risco de cada conversa de teste, e a fila. É daqui que o painel do Looker Studio vai ler.

In [17]:
# ATENCAO: nomes proprios de propósito.
# O arquivo que a camada gold consome (tb_score_risco) e gerado pelo
# 09_modelo_risco_bot.py, com mais colunas (id_cliente, contatos_faixa,
# reabertura, quintil) e a coluna chamada faixa_operacional.
# O notebook grava a SUA propria copia, para nao sobrescrever aquele arquivo.
saida = T[["id_conversa","categoria_assunto","segmento_cliente","risco","faixa","falhou"]]
saida.to_csv("score_risco_notebook.csv", index=False, sep=";", encoding="utf-8-sig")
fila.to_csv("fila_correcao_notebook.csv", index=False, sep=";", encoding="utf-8-sig")

print(f"score_risco_notebook.csv    — {len(saida):,} linhas".replace(",", "."))
print(f"fila_correcao_notebook.csv  — {len(fila)} assuntos")
print("\nOs arquivos que a gold consome continuam sendo os do 09_modelo_risco_bot.py.")


score_risco_notebook.csv    — 7.712 linhas
fila_correcao_notebook.csv  — 10 assuntos

Os arquivos que a gold consome continuam sendo os do 09_modelo_risco_bot.py.


---

## Resumo, para quem for apresentar

| | |
|---|---|
| **O que o modelo faz** | Estima, na abertura, a chance de a Kenzie não resolver |
| **Com o que ele decide** | 12 colunas conhecidas no primeiro minuto |
| **Como foi avaliado** | Em clientes que ele nunca viu |
| **Por que regressão logística** | Cada característica sai com peso visível; a random forest não melhorou |
| **O que fazer com a nota** | Encaminhar direto os de maior risco — onde cortar é decisão de negócio |

**A afirmação que não podemos fazer:** que a V2 resolve mais que a V1. Encaminhar cedo não deixa a Kenzie mais inteligente — deixa ela sabendo a hora de passar a bola.